<a href="https://colab.research.google.com/github/harinijk/CourseScheduler/blob/main/4511W_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import time

random.seed(42)

def make_section(section_id, days, start, end, credits, is_online):
    s = {}
    s['section_id'] = section_id
    s['days']= days
    s['start'] = start
    s['end'] = end
    s['credits']= credits
    s['is_online']= is_online
    return s

def mins_to_str(m):
    hours = m // 60
    minutes = m % 60
    if hours < 12:
        suffix = 'AM'
    else:
        suffix = 'PM'
    h12 = hours % 12
    if h12 == 0:
        h12 = 12
    return str(h12) + ':' + str(minutes).zfill(2) + ' ' + suffix

def print_section(s):
    days_str = ''
    for d in s['days']:
        days_str = days_str + d
    if s['is_online'] == True:
        online_str = 'Online'
    else:
        online_str = 'In-person'
    print(s['section_id'], '|', days_str, '|',
          mins_to_str(s['start']), '-', mins_to_str(s['end']),
          '|', s['credits'], 'credits |', online_str)

test = make_section('CS101-001', ['M','W','F'], 540, 590, 3, False)
print_section(test)

In [ ]:
def sections_conflict(s1, s2):
    if s1['is_online'] == True:
        return False
    if s2['is_online'] == True:
        return False

    share_day = False
    for d1 in s1['days']:
        for d2 in s2['days']:
            if d1 == d2:
                share_day = True

    if share_day == False:
        return False

    if s1['start'] < s2['end'] and s2['start'] < s1['end']:
        return True
    return False


def is_consistent(assignment, val, max_credits):
    for course in assignment:
        if sections_conflict(val, assignment[course]) == True:
            return False

    total = val['credits']
    for course in assignment:
        total = total + assignment[course]['credits']
    if total > max_credits:
        return False

    return True


# Test
s_a = make_section('A', ['M','W'], 540, 640, 3, False)
s_b = make_section('B', ['M','W'], 600, 700, 3, False)
s_c = make_section('C', ['T','R'], 540, 640, 3, False)

print('A and B conflict (expect True):', sections_conflict(s_a, s_b))
print('A and C conflict (expect False):', sections_conflict(s_a, s_c))

In [ ]:
def build_graph(variables):
    graph = {}
    for v in variables:
        graph[v] = []
    for i in range(len(variables)):
        for j in range(len(variables)):
            if i != j:
                v1 = variables[i]
                v2 = variables[j]
                if v2 not in graph[v1]:
                    graph[v1].append(v2)
    return graph


def revise(domains, xi, xj):
    revised = False
    new_domain = []
    for vi in domains[xi]:
        has_support = False
        for vj in domains[xj]:
            if sections_conflict(vi, vj) == False:
                has_support = True
        if has_support == True:
            new_domain.append(vi)
        else:
            revised = True
    domains[xi] = new_domain
    return revised


def ac3(domains, graph):
    queue = []
    for xi in graph:
        for xj in graph[xi]:
            queue.append((xi, xj))

    while len(queue) > 0:
        xi = queue[0][0]
        xj = queue[0][1]
        queue = queue[1:]

        if revise(domains, xi, xj) == True:
            if len(domains[xi]) == 0:
                return False
            for xk in graph[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True


In [ ]:
def copy_domains(domains):
    new_domains = {}
    for v in domains:
        new_domains[v] = []
        for s in domains[v]:
            new_domains[v].append(s)
    return new_domains


def lcv_score(var, val, domains, graph, assignment):
    count = 0
    for nb in graph[var]:
        if nb not in assignment:
            for nv in domains[nb]:
                if sections_conflict(val, nv) == True:
                    count = count + 1
    return count


def backtrack(variables, domains, assignment, graph,
              nodes, backtracks, use_mrv, use_lcv, use_mac,
              max_credits, solutions, max_solutions, start_time, timeout):

    if time.time() - start_time > timeout:
        return nodes, backtracks
    if len(solutions) >= max_solutions:
        return nodes, backtracks

    unassigned = []
    for v in variables:
        if v not in assignment:
            unassigned.append(v)

    if len(unassigned) == 0:
        sol = {}
        for k in assignment:
            sol[k] = assignment[k]
        solutions.append(sol)
        return nodes, backtracks

    if use_mrv == True:
        best_var   = unassigned[0]
        best_size  = len(domains[unassigned[0]])
        for v in unassigned:
            if len(domains[v]) < best_size:
                best_size = len(domains[v])
                best_var  = v
        var = best_var
    else:
        var = unassigned[0]


    if use_lcv == True:
        scores = []
        for val in domains[var]:
            sc = lcv_score(var, val, domains, graph, assignment)
            scores.append((sc, val))
        scores.sort(key=lambda x: x[0])
        ordered_vals = []
        for pair in scores:
            ordered_vals.append(pair[1])
    else:
        ordered_vals = []
        for val in domains[var]:
            ordered_vals.append(val)


    for val in ordered_vals:
        nodes = nodes + 1
        if is_consistent(assignment, val, max_credits) == True:
            assignment[var] = val

            if use_mac == True:
                saved = copy_domains(domains)
                domains[var] = [val]
                feasible = ac3(domains, graph)
            else:
                feasible = True
                saved = None

            if feasible == True:
                nodes, backtracks = backtrack(
                    variables, domains, assignment, graph,
                    nodes, backtracks, use_mrv, use_lcv, use_mac,
                    max_credits, solutions, max_solutions, start_time, timeout)

            del assignment[var]
            if use_mac == True and saved is not None:
                for v in saved:
                    domains[v] = saved[v]
            backtracks = backtracks + 1

    return nodes, backtracks


def solve(catalog, mode, max_credits, max_solutions, timeout):
    variables = []
    for course in catalog:
        variables.append(course)

    domains = {}
    for course in catalog:
        domains[course] = []
        for s in catalog[course]:
            domains[course].append(s)

    graph     = build_graph(variables)
    solutions = []
    nodes      = 0
    backtracks = 0
    start_time = time.time()

    if mode == 'full':
        use_mrv = True
        use_lcv = True
        use_mac = True
    elif mode == 'bt+ac3':
        use_mrv = False
        use_lcv = False
        use_mac = True
    else:
        use_mrv = False
        use_lcv = False
        use_mac = False

    if mode == 'bt+ac3' or mode == 'full':
        ok = ac3(domains, graph)
        if ok == False:
            elapsed = time.time() - start_time
            return solutions, nodes, backtracks, elapsed

    nodes, backtracks = backtrack(
        variables, domains, {}, graph,
        nodes, backtracks, use_mrv, use_lcv, use_mac,
        max_credits, solutions, max_solutions, start_time, timeout)

    elapsed = time.time() - start_time
    return solutions, nodes, backtracks, elapsed



In [ ]:
def preference_score(assignment):
    preferred_start = 540
    preferred_end = 1020
    max_gap = 90

    sections = []
    for course in assignment:
        sections.append(assignment[course])

    n = len(sections)
    if n == 0:
        return 0.0

    time_score = 0
    for s in sections:
        if s['start'] >= preferred_start and s['end'] <= preferred_end:
            time_score = time_score + 1.0
        elif s['start'] >= preferred_start or s['end'] <= preferred_end:
            time_score = time_score + 0.5
    time_score = time_score / n


    day_classes = {}
    for s in sections:
        if s['is_online'] == False:
            for d in s['days']:
                if d not in day_classes:
                    day_classes[d] = []
                day_classes[d].append((s['start'], s['end']))

    total_gaps = 0
    big_gaps = 0
    for d in day_classes:
        intervals = day_classes[d]

        for i in range(len(intervals)):
            for j in range(i+1, len(intervals)):
                if intervals[j][0] < intervals[i][0]:
                    temp = intervals[i]
                    intervals[i] = intervals[j]
                    intervals[j] = temp
        for i in range(1, len(intervals)):
            gap = intervals[i][0] - intervals[i-1][1]
            total_gaps = total_gaps + 1
            if gap > max_gap:
                big_gaps = big_gaps + 1

    if total_gaps > 0:
        gap_score = 1.0 - (big_gaps / total_gaps)
    else:
        gap_score = 1.0


    active_days = {}
    for s in sections:
        if s['is_online'] == False:
            for d in s['days']:
                active_days[d] = 1
    day_score = len(active_days) / 5.0

    early_count = 0
    for s in sections:
        if s['start'] < 540:
            early_count = early_count + 1
    early_score = 1.0 - (early_count / n)

    final = (0.4 * time_score + 0.2 * gap_score +
             0.2 * day_score  + 0.2 * early_score) * 100
    return round(final, 2)


# Test
test_assignment = {
    'CS101': make_section('CS101-001', ['M','W','F'], 540, 590, 3, False),
    'MATH':  make_section('MATH-001',  ['T','R'],     660, 750, 4, False),
}
print('Sample score:', preference_score(test_assignment), '/ 100')

In [ ]:
#USED AI TO CREATE EXAMPLES

catalog = {}

catalog['CSCI4041-Algorithms'] = [
    make_section('4041-001', ['M','W','F'], 540,  590,  3, False),
    make_section('4041-002', ['T','R'],     660,  750,  3, False),
    make_section('4041-003', ['M','W','F'], 780,  830,  3, False),
    make_section('4041-004', ['T','R'],     900,  990,  3, False),
    make_section('4041-005', ['M','W'],     1020, 1110, 3, True),
]

catalog['CSCI4061-OS'] = [
    make_section('4061-001', ['M','W','F'], 480,  530,  3, False),
    make_section('4061-002', ['T','R'],     540,  630,  3, False),
    make_section('4061-003', ['M','W','F'], 660,  710,  3, False),
    make_section('4061-004', ['T','R'],     780,  870,  3, False),
    make_section('4061-005', ['M','W'],     960,  1050, 3, False),
]

catalog['WRIT3562-TechWriting'] = [
    make_section('3562-001', ['M','W','F'], 600,  650,  3, False),
    make_section('3562-002', ['T','R'],     720,  810,  3, False),
    make_section('3562-003', ['M','W'],     840,  930,  3, True),
    make_section('3562-004', ['T','R'],     1020, 1110, 3, False),
]

catalog['CSCI4511-AI'] = [
    make_section('4511-001', ['M','W','F'], 540,  590,  3, False),
    make_section('4511-002', ['T','R'],     600,  690,  3, False),
    make_section('4511-003', ['M','W','F'], 720,  770,  3, False),
    make_section('4511-004', ['T','R'],     840,  930,  3, False),
    make_section('4511-005', ['M'],         960,  1110, 3, True),
]

print('Catalog built with', len(catalog), 'courses')
for course in catalog:
    print(' ', course, '-', len(catalog[course]), 'sections')

In [ ]:
#used AI for getting print format


modes = ['bt', 'bt+ac3', 'full']
labels = {'bt': 'BT (baseline)', 'bt+ac3': 'BT + AC3', 'full': 'Full (BT+AC3+MRV+LCV)'}

for mode in modes:

    cat_copy = {}
    for course in catalog:
        cat_copy[course] = []
        for s in catalog[course]:
            cat_copy[course].append(s)

    solutions, nodes, backtracks, elapsed = solve(
        cat_copy, mode=mode, max_credits=16, max_solutions=5, timeout=30)

    for i in range(len(solutions)):
        for j in range(i+1, len(solutions)):
            if preference_score(solutions[j]) > preference_score(solutions[i]):
                temp = solutions[i]
                solutions[i] = solutions[j]
                solutions[j] = temp

    print('====', labels[mode], '====')
    print('Solutions found:', len(solutions))
    print('Nodes expanded:', nodes)
    print('Backtracks:', backtracks)
    print('Time (s):', round(elapsed, 3))

    if len(solutions) > 0:
        best = solutions[0]
        print('Best score:', preference_score(best), '/ 100')
        print('Best schedule:')
        for course in best:
            print('  ', end='')
            print_section(best[course])
    print()

In [ ]:
all_time_slots = [
    (480,  570),
    (540,  630),
    (600,  690),
    (660,  750),
    (720,  810),
    (780,  870),
    (840,  930),
    (900,  990),
    (960,  1050),
    (1020, 1110),
]

all_day_patterns = [
    ['M','W','F'],
    ['T','R'],
    ['M','W'],
    ['M'],
    ['T'],
]

def generate_catalog(n_courses, sections_per_course, seed):
    rng = random.Random(seed)
    cat = {}
    for i in range(n_courses):
        name = 'COURSE_' + str(i+1)
        sections = []
        used = []
        attempts = 0
        while len(sections) < sections_per_course and attempts < 300:
            attempts = attempts + 1
            slot_idx = rng.randint(0, len(all_time_slots) - 1)
            day_idx  = rng.randint(0, len(all_day_patterns) - 1)
            slot = all_time_slots[slot_idx]
            days = all_day_patterns[day_idx]
            key  = str(slot) + str(days)
            if key in used:
                continue
            used.append(key)
            cr_choices = [3, 3, 3, 4]
            cr_idx= rng.randint(0, 3)
            cr = cr_choices[cr_idx]
            online = False
            if rng.random() < 0.1:
                online = True
            sec_id = name + '-' + str(len(sections)+1)
            sections.append(make_section(sec_id, days, slot[0], slot[1], cr, online))
        cat[name] = sections
    return cat

n_courses_list = [3, 4, 5, 6, 7]
n_trials = 10
modes = ['bt', 'bt+ac3', 'full']

result_nodes  = {}
result_scores = {}
result_times  = {}

for n in n_courses_list:
    for mode in modes:
        result_nodes[(n, mode)]  = []
        result_scores[(n, mode)] = []
        result_times[(n, mode)]  = []

for n in n_courses_list:
    for trial in range(n_trials):
        cat = generate_catalog(n, 5, seed=trial*1000+n)
        for mode in modes:
            cat_copy = {}
            for course in cat:
                cat_copy[course] = []
                for s in cat[course]:
                    cat_copy[course].append(s)

            solutions, nodes, backtracks, elapsed = solve(
                cat_copy, mode=mode, max_credits=18, max_solutions=5, timeout=30)

            best_score = 0
            for sol in solutions:
                sc = preference_score(sol)
                if sc > best_score:
                    best_score = sc

            result_nodes[(n, mode)].append(nodes)
            result_scores[(n, mode)].append(best_score)
            result_times[(n, mode)].append(elapsed)

print('\nTable 1: Mean Nodes Expanded')
print('{:>10} {:>12} {:>12} {:>12}'.format('Courses', 'BT', 'BT+AC3', 'Full'))
print('-' * 50)
for n in n_courses_list:
    bt_avg   = sum(result_nodes[(n,'bt')])     / n_trials
    ac3_avg  = sum(result_nodes[(n,'bt+ac3')]) / n_trials
    full_avg = sum(result_nodes[(n,'full')])   / n_trials
    print('{:>10} {:>12.1f} {:>12.1f} {:>12.1f}'.format(n, bt_avg, ac3_avg, full_avg))

print('\nTable 2: Preference Score (5-course instances)')
print('{:>25} {:>10} {:>10}'.format('Solver', 'Mean', 'Std'))
print('-' * 48)
for mode in modes:
    scores = result_scores[(5, mode)]
    mean   = sum(scores) / len(scores)
    var    = 0
    for sc in scores:
        var = var + (sc - mean) ** 2
    var = var / len(scores)
    std = var ** 0.5
    print('{:>25} {:>10.1f} {:>10.1f}'.format(mode, mean, std))

In [ ]:
import matplotlib.pyplot as plt

bt_means = []
ac3_means = []
full_means = []

for n in n_courses_list:
    bt_means.append(  sum(result_nodes[(n,'bt')])     / n_trials)
    ac3_means.append( sum(result_nodes[(n,'bt+ac3')]) / n_trials)
    full_means.append(sum(result_nodes[(n,'full')])   / n_trials)

plt.figure(figsize=(8,5))
plt.plot(n_courses_list, bt_means,   marker='o', color='red',   label='BT')
plt.plot(n_courses_list, ac3_means,  marker='o', color='orange',label='BT+AC3')
plt.plot(n_courses_list, full_means, marker='o', color='green', label='Full')
plt.yscale('log')
plt.xlabel('Number of Courses')
plt.ylabel('Mean Nodes Expanded (log scale)')
plt.title('Search Effort vs Problem Size')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

bt_scores = result_scores[(5, 'bt')]
ac3_scores = result_scores[(5, 'bt+ac3')]
full_scores = result_scores[(5, 'full')]

plt.figure(figsize=(7,5))
plt.boxplot([bt_scores, ac3_scores, full_scores],
            labels=['BT', 'BT+AC3', 'Full'],
            patch_artist=True)
plt.ylabel('Best Preference Score (0-100)')
plt.title('Schedule Quality at 5 Courses')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()